[![Mở trong Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/thanhmk/hocmaytrongtaichinh/blob/main/Module_5/L4.ipynb)

## HỌC MÁY TRONG TÀI CHÍNH
MODULE 5 | BÀI 4


---

# **MẠNG NƠ-RON TRONG TÀI CHÍNH**

|  |  |
|:---|:---|
|**Thời gian đọc** |  25 phút |
|**Kiến thức nền** | Kiến trúc mạng nơ-ron, dữ liệu kinh tế  |
|**Từ khóa** |Keras, TensorFlow, các lớp, sai số bình phương trung bình, sai số tuyệt đối trung bình  |


---

In [ ]:
# Chuẩn bị môi trường trên Colab (chạy ô này một lần trước khi chạy các ô khác)
%pip install -q pandas-datareader yfinance
import os, urllib.request, urllib.parse

BASE = "https://raw.githubusercontent.com/thanhmk/hocmaytrongtaichinh/main/Module_5/"
FILES = ['DJI_FuturesPredict.csv']
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(BASE + urllib.parse.quote(f), f)
        print("Đã tải:", f)

*Bài trước đã nói về cách hoạt động và kiến trúc của mạng nơ-ron. Bài này áp dụng điều đó vào thực tế, cho bạn đọc làm quen thêm với bước chuẩn bị dữ liệu và biên dịch mô hình.*

## **1. Bài toán dự báo**

Trong bài này, ta sẽ xem việc dự báo lợi suất hằng tháng của hợp đồng tương lai Dow Jones Industrial (DJI), cụ thể là hợp đồng tương lai Mini DJI với mã YM=F. Các biến dự báo ta dùng là các chỉ báo kinh tế từ cơ sở dữ liệu kinh tế FRED. Ta xét năm chỉ báo kinh tế, gồm:


1.   Thu nhập trung bình theo giờ của tất cả người lao động
2.   Tất cả người lao động ngành sản xuất
3.   Chỉ số giá sản xuất theo ngành
4.   Đơn đặt hàng mới hàng hóa lâu bền
5.   Đơn đặt hàng mới của nhà sản xuất: Tổng ngành sản xuất

Dữ liệu kinh tế có thể tìm thấy tại trang https://fred.stlouisfed.org/ và truy cập qua khóa API, mà bạn có thể lấy bằng cách đăng ký tài khoản. Ta bắt đầu bằng việc nhập dữ liệu kinh tế bằng gói `pandas_datareader`.

## **2. Lấy dữ liệu**

Ta nhập `pandas_datareader` bên dưới cùng với pandas vì ta sẽ làm việc với DataFrame.

In [ ]:
# pip install pandas_datareader

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd
import pandas_datareader as pdr  # Access FRED

Bây giờ ta đã nhập gói cần thiết để lấy dữ liệu từ FRED, ta chỉ định khóa API đã có sau khi tạo tài khoản. Việc lấy khóa API khá dễ và nhanh.

In [5]:
# extract api key: put your key in between the angle brackets < >
# myKey = "xxx"
#fred_api_key = "<ENTER YOUR API KEY>"
fred_api_key = "c59f93ca4fb1e3ab14bf42595cfa644f"

Ta sẽ dùng hàm tạo bên dưới tên là *get_fred_data*, giúp việc nhập dữ liệu rất đơn giản. Các tham số đầu vào là danh sách từ khóa chỉ báo kinh tế và khoảng thời gian của dữ liệu, tức là ngày bắt đầu và kết thúc.

In [6]:
# Using code from FRED API: Get US Economic Data using Python


def get_fred_data(param_list, start_date, end_date):
    df = pdr.DataReader(param_list, "fred", start_date, end_date)
    return df.reset_index()

Bên dưới là danh sách từ khóa chỉ báo kinh tế cho năm biến dự báo. Các từ khóa này lấy từ trang FRED. Xem ví dụ được khoanh đỏ ở Hình 1 bên dưới.

**Hình 1: Từ khóa của Đơn đặt hàng mới của nhà sản xuất - Tổng ngành sản xuất.**

![](https://drive.google.com/uc?export=view&id=1Cvyx_Le72AbkMkXecnFmAndxb4V5FwIW)

Danh sách các chỉ báo đã dùng và cân nhắc nằm ở ô tiếp theo bên dưới. Ta nhập dữ liệu và xem qua bằng lệnh tail(10) của pandas để xem 10 bản ghi cuối. Lưu ý ta xét dữ liệu hằng tháng và lấy dữ liệu từ tháng 1/2000 đến tháng 4/2022 (bao gồm cả hai mốc).

In [7]:
# PCUOMFGOMFG :        Producer Price Index by Industry - Monthly
# BOGZ1FL073164003Q :  Interest Rates and Price Indexes; NYSE Composite Index, Level-Quarterly
# CES0500000003 :      Average hourly earnings of all employees
# MANEMP:              All Employees Manufacturing
# DGORDER:             Durable goods New Orders
# AMTMNO:              Manufacturers' New Orders: Total Manufacturing
# FEDFUNDS:            Fed fund rate. Lagging indicator


series = ["CES0500000003", "MANEMP", "PCUOMFGOMFG", "DGORDER", "AMTMNO"]
# get data for series
df = get_fred_data(param_list=series, start_date="2000-01-01", end_date="2022-05-03")
df.set_index("DATE", drop=True, inplace=True)
print(df.shape)
df.tail(10)

(269, 5)


,CES0500000003,MANEMP,PCUOMFGOMFG,DGORDER,AMTMNO
DATE,,,,,
2021-08-01,30.78,12353,220.114,264287,534721
2021-09-01,30.98,12393,221.869,259192,533538
2021-10-01,31.14,12453,225.478,263534,546544
2021-11-01,31.23,12498,227.109,268650,554883
2021-12-01,31.39,12537,226.470,277487,566241
2022-01-01,31.60,12564,231.495,280925,571817
2022-02-01,31.64,12601,237.245,277036,571624
2022-03-01,31.80,12674,245.404,277554,583312
2022-04-01,31.96,12726,249.560,279157,586866


Vậy là ta đã có các biến dự báo, hãy lấy biến mục tiêu, là giá đóng cửa điều chỉnh của hợp đồng tương lai Mini DJI. Ta lấy dữ liệu này bằng thư viện yahoo finance (*yfinance*). Ta nhập dữ liệu hằng tháng cho cùng khung thời gian với các biến dự báo. Ta đã nhập Open, High, Low, Close, Adj Close và Volume; tuy nhiên ta chỉ xét Adj Close vì nó có tính đến các sự kiện doanh nghiệp.

In [8]:
import yfinance as yf

# Mini DJI Futures ticker is YM=F
data = yf.download(tickers="YM=F", start="2000-01-01", end="2022-05-03", interval="1mo")
# Print data
print(data.shape)
data.tail(10)

[*********************100%***********************]  1 of 1 completed

(206, 5)


Price,Close,High,Low,Open,Volume
Ticker,YM=F,YM=F,YM=F,YM=F,YM=F
Date,,,,,
2021-06-01,34399.0,34827.0,32902.0,34570.0,2619173
2021-07-01,34832.0,35072.0,33623.0,34448.0,3242166
2021-09-01,33722.0,35504.0,33478.0,35382.0,3555180
2021-10-01,35704.0,35777.0,33383.0,33752.0,3648899
2021-11-01,34457.0,36446.0,34384.0,35796.0,3671355
2021-12-01,36226.0,36572.0,33928.0,34524.0,3248399
2022-01-01,34997.0,36832.0,33031.0,36304.0,5689330
2022-02-01,33840.0,35752.0,32167.0,34915.0,5115575


Ta có các biến dự báo và biến mục tiêu ở các DataFrame khác nhau; tuy nhiên có thể nối/gộp chúng bằng cột chỉ mục ngày chung. Ta làm việc này bằng lệnh merge của pandas. Cũng cần xem các giá trị thiếu hay Null trong tập dữ liệu. Điều này có thể do không có dữ liệu ở những ngày đó. May là cột Thu nhập trung bình theo giờ của tất cả người lao động (CES0500000003) chỉ thiếu 39 dòng, không quá nhiều. Ta chọn cách đơn giản nhất là bỏ chúng vì không có quá nhiều giá trị thiếu. Ta còn lại 166 dòng.

In [9]:
# Make time zones non-timezone aware so as to allow the join
df.index = df.index.tz_localize(None)
data.index = data.index.tz_localize(None)

In [13]:
df2predict = pd.merge(df, data["Close"], left_index=True, right_index=True)
print(df2predict.tail())
print(len(df2predict))
df2predict.isnull().sum()
df2predict = df2predict.dropna()

print("Print rows remaining after removed missing values {}".format(len(df2predict)))

            CES0500000003  MANEMP  PCUOMFGOMFG  DGORDER  AMTMNO     YM=F
2021-12-01          31.39   12537      226.470   277487  566241  36226.0
2022-01-01          31.60   12564      231.495   280925  571817  34997.0
2022-02-01          31.64   12601      237.245   277036  571624  33840.0
2022-03-01          31.80   12674      245.404   277554  583312  34618.0
2022-04-01          31.96   12726      249.560   279157  586866  32882.0
206
Print rows remaining after removed missing values 166


Ta đã làm khá nhiều việc để dựng tập dữ liệu này, nên hãy lưu thành csv để dễ đọc lại khi cần làm việc. Chỉ định đường dẫn lưu và tên; trong trường hợp của mình, mình đặt tên là `DJI_FuturesPredict.csv`. Khi muốn đọc dữ liệu vào, ta chỉ cần dùng lệnh `read_csv` của pandas như ở ô bên dưới.

In [14]:
# Store to csv
#path2copy = "../../data"
df2predict.to_csv("DJI_FuturesPredict.csv", index=True, index_label="Date")

In [15]:
# read in pre-stored data

df2predict = pd.read_csv("DJI_FuturesPredict.csv")
df2predict.set_index("Date", drop=True, inplace=True)

# Quick check that the data looks familiar.
df2predict.tail()

,CES0500000003,MANEMP,PCUOMFGOMFG,DGORDER,AMTMNO,YM=F
Date,,,,,,
2021-12-01,31.39,12537,226.470,277487,566241,36226.0
2022-01-01,31.60,12564,231.495,280925,571817,34997.0
2022-02-01,31.64,12601,237.245,277036,571624,33840.0
2022-03-01,31.80,12674,245.404,277554,583312,34618.0
2022-04-01,31.96,12726,249.560,279157,586866,32882.0


## **3. Phân tích và chuẩn bị dữ liệu**

Mạng nơ-ron nhạy với thang đo của các đặc trưng, cả về độ chính xác lẫn tốc độ huấn luyện. Không có một cách co giãn dữ liệu nào là tốt nhất, vì ta có thể dùng phần trăm thay đổi, min-max hoặc chuẩn hóa chuẩn, cùng nhiều cách khác, và tùy thuộc vào bài toán. Trong ví dụ này, ta dùng phương pháp co giãn min-max của `sklearn`. Công thức là:
$$
\begin{align}
x_{scaled} = \frac{x - x_{min}}{x_{max}-x_{min}}
\end{align}
$$
trong đó,

*   $x_{min}$ = giá trị đặc trưng nhỏ nhất
*   $x_{max}$ = giá trị đặc trưng lớn nhất

In [16]:
# Feature scaling
from sklearn.preprocessing import MinMaxScaler

# scale features
scaler = MinMaxScaler()
scale_model = scaler.fit(df2predict[series])
df2predict[series] = scale_model.transform(df2predict[series])
df2predict.tail()

,CES0500000003,MANEMP,PCUOMFGOMFG,DGORDER,AMTMNO,YM=F
Date,,,,,,
2021-12-01,0.952181,0.406547,0.755816,0.852189,0.918290,36226.0
2022-01-01,0.969799,0.416051,0.808957,0.874648,0.940380,34997.0
2022-02-01,0.973154,0.429074,0.869765,0.849243,0.939616,33840.0
2022-03-01,0.986577,0.454769,0.956049,0.852627,0.985920,34618.0
2022-04-01,1.000000,0.473073,1.000000,0.863099,1.000000,32882.0


Ta quan tâm đến việc dự báo lợi suất của hợp đồng tương lai DJI. Do đó, ta chuyển cột giá Adj Close thành lợi suất phần trăm.

In [20]:
# Use percent change instead of actual values
# If all columns are transformed to a % change.
# df2predict = df2predict.pct_change()

# % change for just the target column
df2predict["Close"] = df2predict["Close"].pct_change()

# Drop any missing values
df2predict.dropna(inplace=True)
# If a time shift is needed because of a lag.
# `df2predict['AdjClose_shift'] = df2predict['Adj Close'].shift(-1)`
# `df2predict.drop('Adj Close', axis=1, inplace=True)`

# Glimpse of data
df2predict.head()

KeyError: 'Close'

Nên xem mối quan hệ giữa các biến dự báo và lợi suất Adj Close. Ta có thể làm bằng hình ảnh vì không có quá nhiều biến dự báo. Để tránh biểu đồ quá rối và xác định độ trễ thời gian giữa lợi suất Adj Close và biến dự báo, ta chọn một mẫu điểm, giả sử 3 năm hay 36 tháng. Hình bên dưới không cho thấy tương quan mạnh giữa các biến dự báo và lợi suất; tuy nhiên đợt giảm vào tháng 2 và tháng 3 năm 2020 xuất hiện ở hầu hết các biến dự báo cũng như ở lợi suất.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
target = "Adj Close"
n_pts = 36

for j in np.arange(len(series)):
    # Check the lag

    var = series[j]
    # Define Data
    # choose sample of points to avoid clutter in plot
    df2plot = df2predict.iloc[len(df2predict) - n_pts : len(df2predict),]  # noQA E203
    x = df2plot.index
    data_1 = df2plot[var]
    data_2 = df2plot[target]

    # Create Plot

    fig, ax1 = plt.subplots()

    ax1.set_xlabel("Date")
    ax1.set_ylabel(var, color="red")
    ax1.plot(x, data_1, color="red")
    plt.xticks(rotation=90)  # Rotates X-Axis Ticks by 45-degrees
    # Adding Twin Axes

    ax2 = ax1.twinx()
    ax2.plot(x, data_2, color="blue")

    # Add label

    ax2.set_ylabel("Adj Close returns", color="blue")
    ax2.tick_params(axis="y", color="blue")

    # Show plot

    plt.show()

Như vậy, bây giờ ta sẵn sàng tạo mạng nơ-ron bằng TensorFlow và Keras.

## **4. Keras và TensorFlow**

Mạng nơ-ron ban đầu được tạo dễ dàng trong Python nhờ thư viện Keras do Francois Chollet tạo ra. Cuốn sách của ông cũng là nguồn tài liệu rất tốt về mạng nơ-ron và học sâu (Chollet 54); tuy nhiên sách không phải mã nguồn mở.

Google tạo ra thư viện TensorFlow (tf) thực hiện các tính toán cho mạng nơ-ron một cách hiệu quả, nhưng không phải là thư viện thân thiện nhất.

Sau đó Google phát hành TensorFlow 2 tích hợp trực tiếp API Keras và đưa giao diện này thành giao diện mặc định hay tiêu chuẩn cho phát triển học sâu trên nền tảng.

Đó là lý do ta hay thấy tf.keras trong mã học sâu, giúp việc biên dịch và chạy các mô hình này dễ và nhanh hơn nhiều.

In [ ]:
# Begin creating our model by importing TensorFlow
import tensorflow as tf

print(tf.__version__)

Bây giờ ta tách dữ liệu thành tập biến dự báo và cột mục tiêu, lần lượt là x và y.

In [ ]:
X = df2predict[series].values
y = df2predict["Adj Close"].values

Ta thực hiện việc chia huấn luyện/kiểm tra cần thiết cho mô hình dự báo. Trong trường hợp của ta, tỷ lệ chia huấn luyện/kiểm tra là 80/20 và việc chia theo thứ tự thời gian, với dữ liệu gần đây nhất là tập chưa thấy.

In [ ]:
test_sz = 0.2
train_sz = int((1 - test_sz) * len(X))
X_train = X[:train_sz]
y_train = y[:train_sz]
X_test = X[train_sz:]
y_test = y[train_sz:]
len(X_train), len(X_test)

Trong ô bên dưới, ta làm một vài việc để dựng mô hình. Ta đặt hạt giống ngẫu nhiên để có thể tái tạo bản dựng này bất cứ lúc nào. Sau đó ta gọi các đối tượng tf.keras để ghép mô hình, bắt đầu từ *Sequential*. Gọi là tuần tự vì ta có thể thêm các lớp vào mạng nơ-ron lần lượt theo thứ tự. Trong ví dụ đơn giản nhất để minh họa, ta chỉ có một lớp, nhưng sau đó sẽ thêm vào. Đối tượng *tf.keras.Dense* nên quen thuộc từ Module 5, Bài 3, ở chỗ ta chỉ định mọi cạnh đều nối đến các nút tiếp theo. Tham số 1 trong đối tượng Dense(1) chỉ định đầu ra là một giá trị liên tục duy nhất, tức là lợi suất dự báo của hợp đồng tương lai DJI.

Lệnh *model.compile* là nơi ta chỉ định hàm mất mát và bộ tối ưu. Trong ví dụ bên dưới, ta dùng sai số tuyệt đối trung bình làm hàm mất mát để đo dự báo lệch bao xa so với thực tế. Xem danh sách các hàm mất mát được hỗ trợ tại ("Keras Loss Functions").

Bộ tối ưu là SGD hay giảm gradient ngẫu nhiên, nhưng Adam là một lựa chọn phổ biến khác. Thước đo hiệu suất sau mỗi vòng lặp là sai số tuyệt đối trung bình, nhưng ta cũng có thể dùng sai số bình phương trung bình. Xem danh sách các thước đo được hỗ trợ tại ("Keras Metrics").

Bạn sẽ thấy có các đối số được chỉ định trong bộ tối ưu như tốc độ học và động lượng. Tốc độ học chỉ định mức thay đổi của mô hình so với thay đổi của sai số. Tất nhiên ta muốn mô hình học nhanh; tuy nhiên cần nhớ rằng giá trị tốc độ học lớn có thể khiến mô hình học quá nhanh từ một bộ trọng số chưa tối ưu, dẫn đến không hội tụ. Điều này có thể gây ra quá trình học không ổn định. Tốc độ học quá nhỏ sẽ làm tăng thời gian huấn luyện, có thể khiến thuật toán bị kẹt. Thường dùng giá trị tốc độ học từ 0,01 đến 1. Động lượng có thể tăng tốc quá trình huấn luyện để hội tụ với ít epoch hơn. Thường dùng giá trị động lượng từ 0,9 đến 0,99.

Epoch là số lần thuật toán chạy qua tập huấn luyện. Càng nhiều epoch thì mô hình càng cần nhiều thời gian huấn luyện. Với sức tính toán hạn chế, cần cân nhắc điều này vì thời gian cần thiết có thể tăng đáng kể. Một lý do khác ta không thể cứ tăng epoch lên thật cao là điều đó cũng có thể gây quá khớp. Ta chọn 10 ở đây cho mục đích minh họa đơn giản nhưng sẽ tăng lên sau để xem kết quả thay đổi. Lưu ý tốc độ học nhỏ cần nhiều epoch hơn và ngược lại.

Kích thước lô (batch size) là cỡ mẫu được đưa qua mạng một lần để lan truyền. Phép tính gradient sai số dựa trên các mẫu đi qua này. Tất nhiên giá trị này cần nhỏ hơn tổng số quan sát trong dữ liệu huấn luyện. Bộ tối ưu giảm gradient ngẫu nhiên cập nhật trọng số dựa trên gradient sai số. Các kích thước thường gặp là bội của 8. Bạn không muốn kích thước lô chiếm tỷ lệ quá lớn so với số quan sát của tập huấn luyện. Xem ("Batch Size Tutorial") để biết cách chọn kích thước lô.

Cần lưu ý kích thước lô và epoch hoạt động trên toàn bộ tập huấn luyện sẵn có (80% quan sát). Mọi quan sát của tập huấn luyện được lấy mẫu theo kích thước lô và đưa qua thuật toán ở các thời điểm khác nhau.

Bây giờ ta đã chỉ định các siêu tham số cần thiết, ta có thể biên dịch và huấn luyện mô hình bằng model.fit.

In [ ]:
# Build the model
# tf.keras: The Keras API integrated into TensorFlow 2

tf.random.set_seed(42)  # first we set random seed
model = tf.keras.Sequential([tf.keras.layers.Dense(1)])  # The output layer

# We compile the model specifying loss, and optimizer.
model.compile(
    loss=tf.keras.losses.mae,  # Los function is MAE, mean absolute error.
    optimizer=tf.keras.optimizers.SGD(
        learning_rate=0.01, momentum=0.9
    ),  # stochastic Gradient descent Optimizer
    metrics=["mae"],
)  # performance metric is MAE

model.fit(X_train, y_train, epochs=10, batch_size=8)  # epoch and batch size specified

Sau khi huấn luyện mô hình, ta có thể xem hiệu suất. Ta dùng sai số tuyệt đối trung bình và sai số bình phương trung bình vì đang làm việc với biến mục tiêu liên tục. Ta thấy hai thước đo này lần lượt dưới 5% và 0,3%.

In [ ]:
# performance
preds = model.predict(X_test)
mae = tf.metrics.mean_absolute_error(y_true=y_test, y_pred=preds.squeeze()).numpy()
mse = tf.metrics.mean_squared_error(y_true=y_test, y_pred=preds.squeeze()).numpy()
mae, mse

Hãy xem liệu ta có cải thiện được kết quả này bằng nhiều lớp hơn và các lựa chọn siêu tham số khác, như tăng số epoch. Ta cũng chỉ định một hàm kích hoạt, cụ thể là hàm *sigmoid* trong ví dụ này. Ta cũng dùng hàm tối ưu Adam thay vì SGD. Bộ tối ưu SGD nhạy với tốc độ học, và dù có thể cần nhiều vòng lặp, nó vẫn có thể hội tụ khá nhanh. Adam là bộ tối ưu phổ biến và được dùng khá hiệu quả trong các thuật toán học sâu. Không giống SGD, Adam không dùng tốc độ học cố định vì nó thích nghi qua các vòng lặp.

Ta thấy cả hai thước đo MAE và MSE đều giảm khi thêm 2 lớp. Thêm nhiều lớp hơn là bước vào lĩnh vực Học sâu và có ích khi dữ liệu phi tuyến. Hãy nhớ thuật toán trở nên dễ gặp vấn đề gradient biến mất khi thêm nhiều lớp, nhất là khi hàm kích hoạt là hàm sigmoid.

In [ ]:
# Improve our model. More epochs, added extra layers

tf.random.set_seed(42)
model_1 = tf.keras.Sequential(
    [
        tf.keras.layers.Dense(
            8, activation="sigmoid", input_shape=(X_train.shape[1],)
        ),  # added extra layer
        tf.keras.layers.Dense(4, activation="sigmoid"),  # added extra layer
        tf.keras.layers.Dense(1),
    ]
)
model_1.compile(
    loss=tf.keras.losses.mae, optimizer=tf.keras.optimizers.Adam(), metrics=["mae"]
)
model_1.fit(X_train, y_train, epochs=100, batch_size=8, verbose=0)

In [ ]:
# performance
preds = model_1.predict(X_test)
mae = tf.metrics.mean_absolute_error(y_true=y_test, y_pred=preds.squeeze()).numpy()
mse = tf.metrics.mean_squared_error(y_true=y_test, y_pred=preds.squeeze()).numpy()
mae, mse

## **5. Kết luận**
Trong bài này, ta đã áp dụng mạng nơ-ron để dự báo lợi suất hằng tháng của hợp đồng tương lai DJI. Ta dùng các chỉ báo kinh tế từ FRED làm biến dự báo và khám phá các cách nhập và định dạng dữ liệu này cho một kịch bản phân tích dự báo. Khi thiết lập mạng nơ-ron, ta đã bàn về một số siêu tham số cần để biên dịch mạng và thêm nhiều lớp hơn ở lần thử cuối. Việc thêm nhiều lớp là cần thiết khi tạo một mô hình học sâu.

**Tài liệu tham khảo**

- Chollet, Francois. *Deep Learning with Python*. Manning, 2017.

- "Keras Loss Functions." Keras Loss Functions List, www.tensorflow.org/api_docs/python/tf/keras/losses. Accessed 18 June 2022.

- "Keras Metrics." Keras Metrics List, www.tensorflow.org/api_docs/python/tf/keras/metrics. Accessed 18 June 2022.

- "Batch Size Tutorial." Batch Size Tutorial, www.machinelearningmastery.com/how-to-control-the-speed-and-stability-of-training-neural-networks-with-gradient-descent-batch-size. Accessed 18 June 2022.